In [1]:
import pandas as pd
import numpy as np

# 1. Define the holiday dataset (Same as before)
all_holiday_data = {
    'Date': [
        '02.01.2022', '25.01.2022', '14.02.2022', '27.03.2022', '15.04.2022', '17.04.2022', '18.04.2022', '19.06.2022', '31.10.2022', '30.11.2022', '25.12.2022', '26.12.2022', '31.12.2022',
        '02.01.2023', '25.01.2023', '14.02.2023', '19.03.2023', '07.04.2023', '09.04.2023', '10.04.2023', '18.06.2023', '31.10.2023', '30.11.2023', '25.12.2023', '26.12.2023', '31.12.2023',
        '02.01.2024', '25.01.2024', '14.02.2024', '10.03.2024', '29.03.2024', '31.03.2024', '01.04.2024', '16.06.2024', '31.10.2024', '30.11.2024', '25.12.2024', '26.12.2024', '31.12.2024',
        '02.01.2025', '25.01.2025', '14.02.2025', '30.03.2025', '18.04.2025', '20.04.2025', '21.04.2025', '15.06.2025', '31.10.2025', '30.11.2025', '25.12.2025', '26.12.2025', '31.12.2025'
    ],
    'Holiday': [
        '2nd January Bank Holiday', 'Burns Night', "Valentine's Day", "Mother's Day (UK)", 'Good Friday', 'Easter Sunday', 'Easter Monday', "Father's Day", 'Halloween', "St. Andrew's Day", 'Christmas Day', 'Boxing Day', "Hogmanay / New Year's Eve",
        '2nd January Bank Holiday', 'Burns Night', "Valentine's Day", "Mother's Day (UK)", 'Good Friday', 'Easter Sunday', 'Easter Monday', "Father's Day", 'Halloween', "St. Andrew's Day", 'Christmas Day', 'Boxing Day', "Hogmanay / New Year's Eve",
        '2nd January Bank Holiday', 'Burns Night', "Valentine's Day", "Mother's Day (UK)", 'Good Friday', 'Easter Sunday', 'Easter Monday', "Father's Day", 'Halloween', "St. Andrew's Day", 'Christmas Day', 'Boxing Day', "Hogmanay / New Year's Eve",
        '2nd January Bank Holiday', 'Burns Night', "Valentine's Day", "Mother's Day (UK)", 'Good Friday', 'Easter Sunday', 'Easter Monday', "Father's Day", 'Halloween', "St. Andrew's Day", 'Christmas Day', 'Boxing Day', "Hogmanay / New Year's Eve"
    ],
    'Importance': [
        3, 4, 5, 5, 4, 4, 3, 4, 3, 3, 5, 4, 5,
        3, 4, 5, 5, 4, 4, 3, 4, 3, 3, 5, 4, 5,
        3, 4, 5, 5, 4, 4, 3, 4, 3, 3, 5, 4, 5,
        3, 4, 5, 5, 4, 4, 3, 4, 3, 3, 5, 4, 5
    ]
}

df_holidays = pd.DataFrame(all_holiday_data)
df_holidays['Date'] = pd.to_datetime(df_holidays['Date'], format='%d.%m.%Y')

# 2. Create the Master Timeline (Empty DF with all dates and hours)
start_date = '2022-01-01'
end_date = '2025-12-31'
all_days = pd.date_range(start=start_date, end=end_date, freq='D')
business_hours = [f"{h:02d}:00" for h in range(8, 17)] # 08:00 to 17:00

# Create a DF with every combination of Day and Business Hour
master_timeline = pd.DataFrame([
    {'Date': d.strftime('%Y-%m-%d'), 'Time': h}
    for d in all_days for h in business_hours
])

# 3. Add Holiday Flags and Importance
# Merge the holiday list into our timeline
master_timeline['Date_dt'] = pd.to_datetime(master_timeline['Date'])
df_holidays_merged = master_timeline.merge(
    df_holidays, left_on='Date_dt', right_on='Date', how='left', suffixes=('', '_drop')
)

# Rename and clean columns
df_holidays_merged = df_holidays_merged.rename(columns={
    'Holiday': 'holiday_name',
    'Importance': 'holiday_importance'
})
df_holidays_merged['is_holiday'] = df_holidays_merged['holiday_name'].notna().astype(int)
df_holidays_merged['holiday_importance'] = df_holidays_merged['holiday_importance'].fillna(0)
df_holidays_merged['holiday_name'] = df_holidays_merged['holiday_name'].fillna('None')

# 4. Calculate the Countdown (Days Until Next Holiday)
holiday_dates = df_holidays['Date'].unique()
holiday_dates = pd.Series(df_holidays['Date'].unique()).sort_values().values

def get_days_until(current_date):
    future_holidays = holiday_dates[holiday_dates >= current_date]
    if len(future_holidays) > 0:
        return (future_holidays[0] - current_date).days
    return 0

df_holidays_merged['Days_Until_Next_Holiday'] = df_holidays_merged['Date_dt'].apply(get_days_until)

# 5. Finalize Structure and Save
df_holidays_final = df_holidays_merged[[
    'Date_dt', 'is_holiday', 'holiday_importance', 'holiday_name', 'Days_Until_Next_Holiday'
]].copy()

# Rename Date_dt to Date
df_holidays_final = df_holidays_final.rename(columns={'Date_dt': 'Date'})

# Adjust Date to include Time (hour) from master_timeline
df_holidays_final['Date'] = pd.to_datetime(df_holidays_final['Date'].dt.date.astype(str) + ' ' + df_holidays_merged['Time'])

df_holidays_final

,Date,is_holiday,holiday_importance,holiday_name,Days_Until_Next_Holiday
0,2022-01-01 08:00:00,0,0.0,None,1
1,2022-01-01 09:00:00,0,0.0,None,1
2,2022-01-01 10:00:00,0,0.0,None,1
3,2022-01-01 11:00:00,0,0.0,None,1
4,2022-01-01 12:00:00,0,0.0,None,1
...,...,...,...,...,...
13144,2025-12-31 12:00:00,1,5.0,Hogmanay / New Year's Eve,0
13145,2025-12-31 13:00:00,1,5.0,Hogmanay / New Year's Eve,0
13146,2025-12-31 14:00:00,1,5.0,Hogmanay / New Year's Eve,0
13147,2025-12-31 15:00:00,1,5.0,Hogmanay / New Year's Eve,0


In [2]:
df_holidays_final.drop(columns=['holiday_name'], errors='ignore', inplace=True)

In [3]:
df_holidays_final.to_csv('processed_csv/holidays_data_preprocessed.csv', index=False)


In [4]:
def check_missing_dates(df):
    """
    Checks if there are any missing hours (08:00-16:00) in the processed DataFrame.
    """
    if 'Date' not in df.columns:
        print("Error: 'Date' column not found in DataFrame.")
        return
    
    # Ensure Date is datetime
    df_dates = pd.to_datetime(df['Date'])
    
    start_dt = df_dates.min()
    end_dt = df_dates.max()
    
    if pd.isna(start_dt) or pd.isna(end_dt):
        print("Error: Could not determine start or end date.")
        return

    # Generate full range (08:00 to 16:00)
    full_range = pd.date_range(start=start_dt.normalize(), end=end_dt.normalize() + pd.Timedelta(days=1), freq='h')
    full_range = full_range[(full_range.hour >= 8) & (full_range.hour < 17)]
    
    missing = full_range.difference(df_dates)
    
    print(f"\n--- Final Data Check ---")
    print(f"Date Range: {start_dt} to {end_dt}")
    print(f"Expected Rows: {len(full_range)}")
    print(f"Actual Rows: {len(df_dates)}")
    
    if len(missing) > 0:
        print(f"WARNING: There are {len(missing)} missing timestamps!")
        if len(missing) > 10:
            print("First 10 missing:")
            print(missing[:10])
        else:
            print(missing)
    else:
        print("SUCCESS: No missing timestamps found in the 08:00-16:00 range.")

# Run the check
check_missing_dates(df_holidays_final)


--- Final Data Check ---
Date Range: 2022-01-01 08:00:00 to 2025-12-31 16:00:00
Expected Rows: 13149
Actual Rows: 13149
SUCCESS: No missing timestamps found in the 08:00-16:00 range.
